# BERT Regression for Article Bias Prediction

This notebook trains a **BERT-based regression model** to predict continuous political-bias scores directly from article text.

The target values are continuous article scores produced by the pairwise Bradley–Terry ranking stage.

### Model pipeline

`article text → BERT tokenizer → BERT encoder → [CLS] representation → regression head → predicted bias score`

The model uses a train/validation/test split, partial BERT fine-tuning, Huber loss, gradient clipping, and early stopping.


## 1. Install and import dependencies

In [ ]:
!pip install -q torch transformers scikit-learn tqdm


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding
)

from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tqdm import tqdm


## 2. Reproducibility and device

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


## 3. Load ranked article scores

`article_bias_scores.csv` contains the article text and continuous target score generated by the ranking model.


In [ ]:
score_df = pd.read_csv("article_bias_scores.csv").dropna()

texts = score_df["text"].tolist()
targets = score_df["bias_score"].to_numpy(dtype=np.float32)

print(f"Articles loaded: {len(texts):,}")
score_df.head()


## 4. Train / validation / test split

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    texts,
    targets,
    test_size=0.15,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=42
)

scaler = StandardScaler()

y_train_scaled = scaler.fit_transform(
    y_train.reshape(-1, 1)
).flatten()

y_val_scaled = scaler.transform(
    y_val.reshape(-1, 1)
).flatten()

y_test_scaled = scaler.transform(
    y_test.reshape(-1, 1)
).flatten()

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


## 5. Tokenizer and PyTorch dataset

Articles are truncated to 256 tokens. Padding is handled dynamically at the batch level.


In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class ArticleDataset(Dataset):
    def __init__(
        self,
        texts,
        targets,
        tokenizer,
        max_length=256
    ):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": float(self.targets[idx]),
        }


train_ds = ArticleDataset(
    X_train, y_train_scaled, tokenizer
)

val_ds = ArticleDataset(
    X_val, y_val_scaled, tokenizer
)

test_ds = ArticleDataset(
    X_test, y_test_scaled, tokenizer
)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

def collate_fn(features):
    labels = torch.tensor(
        [f["labels"] for f in features],
        dtype=torch.float32
    )

    batch = collator([
        {
            "input_ids": f["input_ids"],
            "attention_mask": f["attention_mask"]
        }
        for f in features
    ])

    batch["labels"] = labels
    return batch


train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn
)


## 6. BERT regression model

The model uses the final hidden-state representation of BERT's `[CLS]` token, followed by dropout and a linear regression head.


In [ ]:
class BertRegressor(nn.Module):
    def __init__(
        self,
        model_name="bert-base-uncased",
        dropout=0.2
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.regressor = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0]
        x = self.dropout(cls)

        return self.regressor(x).squeeze(-1)


model = BertRegressor(
    model_name=model_name
).to(device)

# Freeze most of BERT
for param in model.bert.parameters():
    param.requires_grad = False

# Fine-tune the final two encoder layers
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

if model.bert.pooler is not None:
    for param in model.bert.pooler.parameters():
        param.requires_grad = True


## 7. Optimizer and loss

In [ ]:
optimizer = AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=2e-5,
    weight_decay=0.01
)

criterion = nn.HuberLoss(delta=1.0)


## 8. Training and evaluation functions

In [ ]:
def run_epoch(model, loader, train=False):
    if train:
        model.train()
    else:
        model.eval()

    losses = []
    preds = []
    labels_all = []

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs, labels)

            if train:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                optimizer.step()

        losses.append(loss.item())
        preds.extend(outputs.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return (
        np.mean(losses),
        np.array(preds),
        np.array(labels_all)
    )


## 9. Train with early stopping

The best validation checkpoint is retained, and training stops after two consecutive epochs without improvement.


In [ ]:
epochs = 10
patience = 2

best_val_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(epochs):
    train_loss, _, _ = run_epoch(
        model,
        train_loader,
        train=True
    )

    val_loss, _, _ = run_epoch(
        model,
        val_loader,
        train=False
    )

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        patience_counter = 0

    else:
        patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_state)
model.to(device)


## 10. Test-set performance

In [ ]:
test_loss, preds_scaled, y_true_scaled = run_epoch(
    model,
    test_loader,
    train=False
)

preds = scaler.inverse_transform(
    preds_scaled.reshape(-1, 1)
).flatten()

y_true = scaler.inverse_transform(
    y_true_scaled.reshape(-1, 1)
).flatten()

mse = mean_squared_error(y_true, preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, preds)
r2 = r2_score(y_true, preds)

print("\nTest results")
print(f"Test loss: {test_loss:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R^2:  {r2:.4f}")

for i in range(min(5, len(preds))):
    print(
        f"Pred: {preds[i]:.3f} | "
        f"True: {y_true[i]:.3f}"
    )


## 11. Predicted vs. true scores

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_true, preds, alpha=0.5)

low = min(y_true.min(), preds.min())
high = max(y_true.max(), preds.max())

plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("True Bradley–Terry Score")
plt.ylabel("Predicted Score")
plt.title("BERT Regression: Predicted vs. True Scores")
plt.tight_layout()
plt.show()


## 12. Save evaluation metrics

In [ ]:
metrics = pd.DataFrame([{
    "model": model_name,
    "rmse": rmse,
    "mae": mae,
    "r2": r2
}])

metrics.to_csv(
    "bert_regression_metrics.csv",
    index=False
)

metrics
